# PASSAGE Sample Visualization

This notebook loads a tensor file generated by PASSAGE and displays it as an image with associated metadata.

**Tensor format:** `(H, W, 3)` with channels:
- Channel 0: Normalized elevation `[0, 1]`
- Channel 1: Marker positions (binary)
- Channel 2: Obstacles mask (binary; 1=blocked)

**Path data:** Stored in `metadata.yaml` under `paths.without_obstacles` and `paths.with_obstacles`.

**Obstacle settings:** Stored in `metadata.yaml` under `obstacles`.

**Solver details:** Stored in `metadata.yaml` under `solver` (includes `backend` and `cost_model`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import yaml

# Add src to path for imports
sys.path.insert(0, str(Path("../src").resolve()))

from passage.generate import tensor_to_image

## Configuration

Set the resolution and sample index to visualize.

In [ ]:
# Configuration
RESOLUTION = 1024  # Resolution: 64, 128, 256, 512, 1024, 2048, 4096
SAMPLE_INDEX = 0  # Sample index to visualize

# Build paths
tensor_path = Path(f"../outputs/dataset/{RESOLUTION}/tensors/{SAMPLE_INDEX:010d}.npy")
metadata_path = Path(f"../outputs/dataset/{RESOLUTION}/metadata/{SAMPLE_INDEX:010d}.yaml")

print(f"Tensor path:   {tensor_path}")
print(f"Metadata path: {metadata_path}")
print(f"Tensor exists: {tensor_path.exists()}")
print(f"Metadata exists: {metadata_path.exists()}")

## Load Tensor and Metadata

In [ ]:
# Load tensor
tensor = np.load(tensor_path)
print(f"Tensor shape: {tensor.shape}")
print(f"Tensor dtype: {tensor.dtype}")

# Load metadata
with open(metadata_path) as f:
    metadata = yaml.safe_load(f)

print(f"\nSample index: {metadata['index']}")
print(f"Resolution: {metadata['resolution']}x{metadata['resolution']}")

## Display Metadata

In [ ]:
print("=" * 60)
print("SAMPLE METADATA")
print("=" * 60)

# Source tile info
print(f"\n📍 Source Tile: {metadata['source_tile']['name']}")
print(f"   Center: ({metadata['center']['lat']:.4f}°, {metadata['center']['lon']:.4f}°)")

# Elevation info
print("\n🏔️  Elevation:")
print(f"   Target: {metadata['target_elevation']:.1f} m")
print(f"   Crop range: [{metadata['crop_min_elevation']:.1f}, {metadata['crop_max_elevation']:.1f}] m")
print(f"   Tile range: [{metadata['tile_min_elevation']:.1f}, {metadata['tile_max_elevation']:.1f}] m")

# Markers info
markers = metadata["markers"]
print("\n📌 Markers:")
print(f"   Start: ({markers[0]['i']}, {markers[0]['j']}) at {markers[0]['elevation']:.1f} m")
print(f"   End:   ({markers[1]['i']}, {markers[1]['j']}) at {markers[1]['elevation']:.1f} m")

# Path info
paths = metadata.get("paths", {})
path_wo = paths.get("without_obstacles") or metadata.get("path", {})
path_w = paths.get("with_obstacles", {})
print("\n🛤️  Paths:")
print(f"   Without obstacles: {path_wo.get('length', 'N/A')} pixels")
print(f"   With obstacles:    {path_w.get('length', 'N/A')} pixels")

# Obstacles info
obstacles_info = metadata.get("obstacles", {})
print("\n🧱 Obstacles:")
print(f"   Enabled: {obstacles_info.get('enabled', 'N/A')}")
print(f"   Ratio target: {obstacles_info.get('ratio_target', 'N/A')}")
print(f"   Ratio actual: {obstacles_info.get('ratio_actual', 'N/A')}")

# Solver info
solver_info = metadata.get("solver", {})
print("\n⚙️  Solver:")
print(f"   Backend: {solver_info.get('backend', 'N/A')}")
print(f"   Algorithm: {solver_info.get('name', 'N/A')}")
print(f"   Diagonal: {solver_info.get('diagonal_movement', 'N/A')}")
print(f"   Weight: {solver_info.get('weight', 'N/A')}")
print(f"   Cost model: {solver_info.get('cost_model', 'N/A')}")
print(f"   Heuristic: {solver_info.get('heuristic', 'N/A')}")

# Timestamp
print(f"\n⏰ Generated: {metadata.get('timestamp', 'N/A')}")

## Visualize Sample Image

Convert tensor to image using `tensor_to_image` with path from metadata.

In [ ]:
# Build path pixels from metadata
paths = metadata.get("paths", {})
use_obstacles = bool(metadata.get("obstacles", {}).get("enabled", False))
path_section = paths.get("with_obstacles") if use_obstacles else paths.get("without_obstacles")
if path_section is None:
    path_section = metadata.get("path", {})
path_pixels = [(p["i"], p["j"]) for p in path_section.get("pixels", [])]
print(f"Path pixels: {len(path_pixels)} points")

# Convert tensor to BGR image
bgr_image = tensor_to_image(tensor, path_pixels=path_pixels, metadata_path=metadata_path)

# Convert BGR to RGB for matplotlib
rgb_image = bgr_image[:, :, ::-1]

# Display
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb_image)
ax.set_title(
    f"Sample {metadata['index']} - {metadata['resolution']}x{metadata['resolution']}\n"
    f"Elevation: [{metadata['crop_min_elevation']:.0f}, {metadata['crop_max_elevation']:.0f}] m"
)
ax.axis("off")

# Add legend

legend_elements = [
    Patch(facecolor="blue", label="Elevation (grayscale)"),
    Patch(facecolor="green", label="Markers"),
    Patch(facecolor="red", label="Obstacles/Path"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=10)

plt.tight_layout()
plt.show()

## Individual Channels

Visualize elevation and marker channels separately.

In [ ]:
# Extract channels
elevation = tensor[:, :, 0]
markers_ch = tensor[:, :, 1]
obstacles_ch = tensor[:, :, 2] if tensor.shape[2] > 2 else np.zeros_like(elevation)

# Create path channel from metadata
paths = metadata.get("paths", {})
use_obstacles = bool(metadata.get("obstacles", {}).get("enabled", False))
path_section = paths.get("with_obstacles") if use_obstacles else paths.get("without_obstacles")
if path_section is None:
    path_section = metadata.get("path", {})
path_pixels = [(p["i"], p["j"]) for p in path_section.get("pixels", [])]
path_ch = np.zeros_like(elevation, dtype=np.float32)
for pi, pj in path_pixels:
    path_ch[pi, pj] = 1.0

# Create figure
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Elevation
im0 = axes[0].imshow(elevation, cmap="terrain")
axes[0].set_title(f"Elevation (normalized)\nRange: [{elevation.min():.3f}, {elevation.max():.3f}]")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Markers
axes[1].imshow(markers_ch, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("Markers")
axes[1].axis("off")
# Mark positions
marker_positions = np.argwhere(markers_ch > 0.5)
for idx, pos in enumerate(marker_positions):
    color = "blue" if idx == 0 else "red"
    label = "Start" if idx == 0 else "End"
    axes[1].plot(
        pos[1], pos[0], "o", color=color, markersize=12, markeredgecolor="white", markeredgewidth=2, label=label
    )
axes[1].legend(loc="upper right")

# Obstacles
axes[2].imshow(obstacles_ch, cmap="Reds", vmin=0, vmax=1)
axes[2].set_title("Obstacles")
axes[2].axis("off")

# Path
axes[3].imshow(path_ch, cmap="Reds", vmin=0, vmax=1)
axes[3].set_title(f"Path ({len(path_pixels)} pixels)")
axes[3].axis("off")

plt.tight_layout()
plt.show()

# Create a full visualization using tensor_to_image (external extraction of path pixels and elevation bounds)
try:
    elev_range = (
        (metadata.get("crop_min_elevation"), metadata.get("crop_max_elevation"))
        if "crop_min_elevation" in metadata and "crop_max_elevation" in metadata
        else None
    )
except Exception:
    elev_range = None

tensor_vis = np.stack([elevation, markers_ch, obstacles_ch], axis=-1)
# pass path_pixels explicitly
bgr_image = tensor_to_image(tensor_vis, path_pixels=path_pixels, elev_range=elev_range)
plt.figure(figsize=(6, 6))
plt.imshow(bgr_image[:, :, ::-1])
plt.title("tensor_to_image composite visualization")
plt.axis("off")
plt.show()

## Elevation Profile Along Path

In [ ]:
if len(path_pixels) > 0:
    # Get elevation values along path (denormalize using crop range)
    crop_min = metadata["crop_min_elevation"]
    crop_max = metadata["crop_max_elevation"]

    path_elevations = []
    for pi, pj in path_pixels:
        norm_elev = elevation[pi, pj]
        real_elev = norm_elev * (crop_max - crop_min) + crop_min
        path_elevations.append(real_elev)

    # Plot profile
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(path_elevations, "b-", linewidth=1)
    ax.fill_between(range(len(path_elevations)), path_elevations, alpha=0.3)

    # Mark start and end
    ax.axhline(y=markers[0]["elevation"], color="green", linestyle="--", label=f"Start: {markers[0]['elevation']:.0f}m")
    ax.axhline(y=markers[1]["elevation"], color="red", linestyle="--", label=f"End: {markers[1]['elevation']:.0f}m")

    ax.set_xlabel("Path Position (pixels)")
    ax.set_ylabel("Elevation (m)")
    ax.set_title("Elevation Profile Along Path")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Statistics
    print("Path elevation stats:")
    print(f"  Min: {min(path_elevations):.1f} m")
    print(f"  Max: {max(path_elevations):.1f} m")
    print(f"  Mean: {np.mean(path_elevations):.1f} m")
    print(
        f"  Total climb: {sum(max(0, path_elevations[i + 1] - path_elevations[i]) for i in range(len(path_elevations) - 1)):.1f} m"
    )
else:
    print("No path data available")